# 00 — Análise do alvo: o gate empírico

Este notebook é **ponto de decisão bloqueante**. As trilhas de modelagem não
devem rodar antes de ele estar executado e com a seção de decisão preenchida.

Ele existe porque escolhas do projeto foram feitas por hipótese e precisavam de
evidência antes de virarem compromisso. Cinco fecharam; a sexta está aberta.

| # | Pergunta | Decisão |
|---|---|---|
| 1 | Equipamentos é mesmo o melhor alvo? | D-01, D-18 |
| 2 | As chaves naturais declaradas são únicas? | semântica de `alterada` em `src/etl/changes.py` |
| 3 | A densidade anual é suficiente? | D-04, D-10 |
| 4 | As coordenadas dão para a trilha geográfica? | D-15, D-17, D-22 |
| 5 | Que colunas o filtro empírico rejeita em toda a série? | D-06 |
| 6 | A regressão da quantidade sustenta uma tarefa secundária? | D-02, **D-37** |

**Recorte:** este notebook roda sobre o **estado de São Paulo** (D-21). Rodou
antes sobre o município apenas, e a mudança de recorte alterou uma das
conclusões — ver o veredito 1. Trocar `RECORTE` abaixo recupera qualquer outro
recorte, porque ele é um prefixo de código IBGE.

Cada seção termina com um **veredito** escrito. Um número sem veredito não fecha
a decisão.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import schema
from src.config.paths import PRIMARY_FOLDER
from src.etl import changes
from src.etl.extract import PERIODOS_ANUAIS
from src.ml.graph import RECORTE_PADRAO, filtro_recorte_sql

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RECORTE = RECORTE_PADRAO          # '35' = estado de SP; '355030' = capital
FILTRO = filtro_recorte_sql(RECORTE)

PERIODOS = changes.periodos_disponiveis()
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")

print(f"Recorte: {RECORTE!r}  ->  {FILTRO}")
print(f"Snapshots na camada primária: {PERIODOS}")
print(f"Transições: {[str(t) for t in changes.transicoes(PERIODOS)]}")
print(f"Tabelas no escopo: {len(schema.FACT_TABLES)}")

# A checagem que faltava quando D-17 foi escrita com dados parciais: afirmar o
# número de snapshots antes de medir qualquer coisa. Ver D-22. O esperado vem de
# `PERIODOS_ANUAIS`, não de um literal — a série cresce um janeiro por ano
# (D-29), e um `9` cravado aqui passaria a mentir em silêncio.
assert len(PERIODOS) >= 2, "rode `python -m src.etl.pipeline` antes deste notebook"
faltando = [p for p in PERIODOS_ANUAIS if p not in PERIODOS]
if faltando:
    print(f"\n[ATENÇÃO] {len(PERIODOS)} de {len(PERIODOS_ANUAIS)} snapshots "
          f"convertidos; faltam {faltando}. Qualquer número abaixo é parcial — "
          "foi exatamente assim que D-17 registrou um teto de cobertura errado.")
extra = [p for p in PERIODOS if p not in PERIODOS_ANUAIS]
if extra:
    print(f"\n[NOTA] snapshots fora da amostra canônica presentes: {extra}")


## 1. Qual tabela é o melhor alvo?

D-01 recomendou `rlEstabEquipamento` com base em duas competências e em contagens
**nacionais**. Aqui a comparação é feita sobre a série inteira e dentro do
recorte de fato, porque um rótulo pode ser denso no país e esparso na amostra.

Quatro candidatas, escolhidas por serem tabelas de fato ligadas ao
estabelecimento cujo conteúdo varia no tempo.

In [ ]:
CANDIDATAS = {
    "rlEstabEquipamento": "co_equipamento",
    "rlEstabComplementar": "co_leito",
    "rlEstabServClass": "co_servico",
    "rlEstabInstFisiAssist": "co_instalacao",
}

def cobertura(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        fato = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not fato.exists() or not raiz.exists():
            continue
        colunas = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{fato}')").fetchall()}
        if col_item not in colunas:
            linhas.append({"periodo": periodo, "erro": f"sem coluna {col_item}"})
            continue
        linhas.append(con.execute(f'''
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                WHERE {FILTRO}
            )
            SELECT '{periodo}' AS periodo,
                   (SELECT COUNT(*) FROM sel) AS estabelecimentos,
                   COUNT(*) AS linhas,
                   COUNT(DISTINCT f.co_unidade) AS com_registro,
                   COUNT(DISTINCT f."{col_item}") AS itens_distintos
            FROM read_parquet('{fato}') f JOIN sel USING (co_unidade)
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if "com_registro" in df:
        df["cobertura_%"] = (100 * df["com_registro"] / df["estabelecimentos"]).round(1)
    return df

for tabela, col in CANDIDATAS.items():
    print(f"\n{'=' * 78}\n{tabela}  (item = {col})\n{'=' * 78}")
    print(cobertura(tabela, col).to_string(index=False))

### O espaço de rótulos de cada candidata

Cobertura alta não basta. O que decide é quantos **eventos de aquisição** cada
candidata gera, porque é o que o modelo tem para aprender, e qual prevalência
resulta — muito baixa torna a tarefa estatisticamente frágil mesmo com muitos
eventos absolutos.

In [ ]:
def espaco_de_rotulos(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for t in changes.transicoes(PERIODOS):
        a = PRIMARY_FOLDER / t.origem / f"{tabela}.parquet"
        b = PRIMARY_FOLDER / t.destino / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / t.origem / "tbEstabelecimento.parquet"
        if not (a.exists() and b.exists() and raiz.exists()):
            continue
        linhas.append(con.execute(f'''
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                WHERE {FILTRO}
            ),
            itens AS (
                SELECT DISTINCT "{col_item}" FROM read_parquet('{b}')
                WHERE "{col_item}" IS NOT NULL
            ),
            tinha AS (
                SELECT DISTINCT co_unidade, "{col_item}"
                FROM read_parquet('{a}') JOIN sel USING (co_unidade)
            ),
            tem AS (
                SELECT DISTINCT co_unidade, "{col_item}"
                FROM read_parquet('{b}') JOIN sel USING (co_unidade)
            ),
            candidatos AS (
                SELECT s.co_unidade, i."{col_item}"
                FROM sel s CROSS JOIN itens i
                EXCEPT SELECT * FROM tinha
            )
            SELECT '{t.destino}' AS transicao,
                   (SELECT COUNT(*) FROM itens) AS itens,
                   COUNT(*) AS candidatos,
                   COUNT(m.co_unidade) AS aquisicoes
            FROM candidatos c
            LEFT JOIN tem m USING (co_unidade, "{col_item}")
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if "aquisicoes" in df:
        df["prevalencia_%"] = (100 * df["aquisicoes"] / df["candidatos"]).round(4)
    return df

resumos = {t: espaco_de_rotulos(t, c) for t, c in CANDIDATAS.items()}
for tabela, df in resumos.items():
    print(f"\n{'=' * 78}\n{tabela}\n{'=' * 78}")
    print(df.to_string(index=False))

In [ ]:
comparacao = pd.DataFrame([
    {
        "tabela": tabela,
        "itens": int(df["itens"].max()),
        "candidatos": int(df["candidatos"].sum()),
        "aquisicoes": int(df["aquisicoes"].sum()),
        "prevalencia_mediana_%": round(float(df["prevalencia_%"].median()), 4),
    }
    for tabela, df in resumos.items()
    if "aquisicoes" in df and not df.empty
]).sort_values("aquisicoes", ascending=False)

print(comparacao.to_string(index=False))

> **Veredito 1 — alvo. `rlEstabEquipamento` mantido, mas o argumento mudou.**
>
> No recorte estadual, somando as oito transições:
>
> | Tabela | Itens | Candidatos | Aquisições | Prevalência mediana |
> |---|---|---|---|---|
> | `rlEstabServClass` | 72 | 55.138.323 | **36.370** | 0,0512% |
> | `rlEstabEquipamento` | 99 | 73.446.373 | **34.571** | 0,0465% |
> | `rlEstabInstFisiAssist` | 51 | 36.881.306 | 19.914 | 0,0473% |
> | `rlEstabComplementar` | 69 | 53.687.388 | 2.718 | 0,0050% |
>
> **A expansão do recorte inverteu a liderança.** No município, equipamentos
> venciam com folga: 12.081 aquisições contra 8.992 de `rlEstabServClass`. No
> estado, serviços passam à frente por 5%. Isso não é ruído — é um efeito de
> composição: o estado tem proporcionalmente muito mais unidades pequenas, que
> registram serviço especializado com mais frequência do que adquirem
> equipamento.
>
> **Equipamentos permanece o alvo**, por três razões que não são o volume:
>
> 1. A pergunta de pesquisa é sobre **recurso físico**. Equipamento é um bem de
>    capital com custo, prazo de aquisição e sentido claro de escassez. Serviço
>    especializado é uma classificação administrativa, e "adquirir um serviço"
>    pode significar apenas reclassificar o que já se fazia.
> 2. Vocabulário maior — 99 tipos contra 72 — dá mais o que ranquear por
>    estabelecimento, que é onde a métrica de destaque mede (D-19).
> 3. A diferença de 5% em volume não compensa trocar um alvo interpretável por
>    um ambíguo.
>
> `rlEstabServClass` fica registrada como **alvo alternativo de primeira
> escolha**, agora com evidência de que sustentaria o experimento. Leitos
> continuam descartados: 2.718 eventos em oito transições, uma ordem de grandeza
> abaixo, e prevalência dez vezes menor.
>
> Registrado em D-18, revisado por D-21.
>
> **Reexecutado em 2026-07-26, com 202601 na série (D-29).** Nove transições em
> vez de oito. A ordem não muda e o argumento não muda:
>
> | Tabela | Itens | Candidatos | Aquisições | Prevalência |
> |---|---|---|---|---|
> | `rlEstabServClass` | 72 | 64.849.904 | **42.208** | 0,0651% |
> | `rlEstabEquipamento` | 99 | 86.654.648 | **40.880** | 0,0472% |
> | `rlEstabInstFisiAssist` | 56 | 44.313.553 | 24.021 | 0,0542% |
> | `rlEstabComplementar` | 69 | 63.100.217 | 3.062 | 0,0049% |
>
> A transição nova, 202501 -> 202601, dá 6.309 aquisições de equipamento — em
> linha com as anteriores, sem salto. Serviços continua à frente em volume por
> cerca de 3%, e equipamentos continua o alvo pelas três razões acima, nenhuma
> delas de volume. `rlEstabInstFisiAssist` passou de 51 para 56 itens: o
> vocabulário de instalações cresceu, o de equipamentos não.


## 2. As chaves naturais declaradas são únicas?

`docs/01-selecao-tabelas.md` declara chave natural para `rlEstabEquipamento` e
`rlEstabComplementar` como **hipótese derivada do dicionário**.

Não é detalhe: sem chave única, `src/etl/changes.py` não distingue modificação de
remoção seguida de inserção, e a taxa de mudança sai inflada — cada alteração
conta duas vezes.

In [ ]:
def unicidade(tabela: str) -> pd.DataFrame:
    chave = schema.CNES_NATURAL_KEY.get(tabela)
    if not chave:
        return pd.DataFrame([{"tabela": tabela, "obs": "sem chave natural declarada"}])
    cols = ", ".join(f'"{c}"' for c in chave)
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        if not p.exists():
            continue
        r = con.execute(f'''
            SELECT COUNT(*) AS linhas,
                   COUNT(*) - COUNT(DISTINCT ({cols})) AS duplicadas
            FROM read_parquet('{p}')
        ''').df().iloc[0]
        linhas.append({
            "tabela": tabela, "periodo": periodo, "chave": " + ".join(chave),
            "linhas": int(r["linhas"]), "duplicadas": int(r["duplicadas"]),
        })
    return pd.DataFrame(linhas)

for tabela in schema.CNES_NATURAL_KEY:
    print(unicidade(tabela).to_string(index=False), "\n")

In [ ]:
# Se houvesse duplicadas: que coluna a mais resolveria?
def chave_minima(tabela: str, periodo: str) -> pd.DataFrame:
    p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
    if not p.exists():
        return pd.DataFrame()
    base = list(schema.CNES_NATURAL_KEY.get(tabela, ()))
    extras = [c for c in schema.CNES_EXTRACT_COLUMNS[tabela]
              if c not in base and not c.startswith("to_char")]
    linhas = []
    for extra in [None, *extras]:
        cols = base + ([extra] if extra else [])
        if not cols:
            continue
        expr = ", ".join(f'"{c}"' for c in cols)
        dup = con.execute(f'''SELECT COUNT(*) - COUNT(DISTINCT ({expr}))
                              FROM read_parquet('{p}')''').fetchone()[0]
        linhas.append({"acrescentando": extra or "(chave declarada)",
                       "duplicadas": int(dup)})
    return pd.DataFrame(linhas).sort_values("duplicadas")

for tabela in schema.CNES_NATURAL_KEY:
    print(f"\n{tabela} em {PERIODOS[-1]}:")
    print(chave_minima(tabela, PERIODOS[-1]).to_string(index=False))

> **Veredito 2 — chaves naturais. As duas hipóteses estavam certas.**
>
> Zero duplicatas em todos os snapshots, para `rlEstabEquipamento` por
> (`co_unidade`, `co_equipamento`, `co_tipo_equipamento`, `tp_sus`) e
> `rlEstabComplementar` por (`co_unidade`, `co_leito`, `co_tipo_leito`).
>
> Deixam de ser hipóteses derivadas do dicionário e passam a fato verificado. A
> classificação `alterada` de `src/etl/changes.py` é confiável para essas duas
> tabelas — e **só** para elas. As outras 42 caem no modo sem chave, em que cada
> modificação conta como remoção mais inserção.
>
> **Revisado em 2026-07-26 (D-27).** As duas conclusões acima valem, mas as duas
> chaves estavam **infladas**: `(co_unidade, co_equipamento, co_tipo_equipamento)`
> e `(co_unidade, co_leito)` já são únicas, e coincidem com a PK do dicionário.
> `tp_sus` e `co_tipo_leito` saíram — senão uma linha que só troca a
> disponibilidade SUS conta como remoção mais inserção, o defeito que a chave
> natural existe para evitar.
>
> E o "só para elas" acabou: agora são **41 chaves declaradas de 44**, todas com
> zero duplicatas nos nove snapshots. Vieram da PK composta do dicionário (25
> tabelas), da chave primária de uma coluna quando ela é de fato única (9, entre
> elas `tbEstabelecimento`) e da busca da menor combinação única quando o
> dicionário citava coluna com outro nome no CSV (5).
>
> Faltam três: `rlEstabServClass` e `rlEstabSipac`, onde nenhuma combinação de até
> quatro colunas materializadas identifica a linha, e `rlMunUnidAcolhim`, que
> precisa de `sq_acolhimento` — hoje `descartada`, logo fora do Parquet e não
> testável. `rlEstabServClass` importa: é o alvo alternativo de primeira escolha
> do veredito 1, e não tem identidade de linha.


## 3. A densidade anual é suficiente?

D-04 fixou snapshots anuais provisoriamente — nove então, dez desde D-29. D-10
pergunta se um ano de
intervalo esconde ciclos.

O sinal: taxa muito alta significa que o intervalo agrega eventos que se queria
separar. Taxa estável e moderada significa que o intervalo está adequado.

In [ ]:
try:
    taxa = changes.taxa_de_mudanca()
except FileNotFoundError as e:
    print(e)
    taxa = None

if taxa is not None:
    foco = taxa[taxa["tabela"].isin(CANDIDATAS)]
    print(foco[["tabela", "periodo_destino", "linhas_origem", "linhas_destino",
                "inserida", "removida", "alterada", "taxa_mudanca",
                "chave_declarada"]].to_string(index=False))

In [ ]:
if taxa is not None and not taxa.empty:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for tabela, grupo in taxa[taxa["tabela"].isin(CANDIDATAS)].groupby("tabela"):
        estilo = "-o" if grupo["chave_declarada"].all() else "--x"
        ax.plot(grupo["periodo_destino"], grupo["taxa_mudanca"], estilo, label=tabela)
    ax.set_title("Taxa de mudança anual por tabela candidata")
    ax.set_xlabel("transição (período de destino)")
    ax.set_ylabel("eventos / linhas")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Linha tracejada = sem chave natural declarada: cada modificação conta")
    print("como remoção mais inserção, então a taxa está superestimada.")

> **Veredito 3 — densidade de snapshots. Anual mantida.**
>
> Taxa de mudança de `rlEstabEquipamento` nas oito transições: 0,112, 0,094,
> 0,101, 0,100, 0,095, 0,085, 0,082, 0,089. Mediana **0,094**, amplitude inteira
> entre 0,082 e 0,112, sem pico nas transições de pandemia.
>
> Série plana é justamente o que indica que o intervalo não está agregando
> eventos que se queira separar. Não há evidência de ciclo escondido que
> justifique o custo de densificar. **D-10 fechada.**
>
> A mudança é dominada por inserção — de 63 mil a 89 mil por transição contra
> cerca de 10 mil remoções — coerente com o crescimento de 67% que motivou D-01.
>
> Leitura: `tbEstabelecimento` aparece com taxa acima de 1,0 e `alterada` zerada
> porque não tem chave natural declarada. É o comportamento documentado em
> `src/etl/changes.py`, não defeito dos dados.
>
> **Recalculado em 2026-07-26 (D-27).** Com a chave do alvo encurtada e 39 novas
> chaves declaradas, a série passa a 0,110, 0,091, 0,097, 0,097, 0,092, 0,083,
> 0,079, 0,087 — mediana **0,091**, ainda plana entre 0,079 e 0,110. O veredito
> não muda.
>
> A leitura sobre `tbEstabelecimento`, sim: ela agora tem chave natural
> (`co_unidade`, única em todos os snapshots), a taxa cai de acima de 1,0 para
> 0,83 na primeira transição e 0,17–0,25 nas seguintes, e `alterada` passa a ser
> preenchida. O artefato que este veredito mandava ler como "comportamento
> documentado" era chave natural faltando, e agora só sobra nas três tabelas
> listadas no veredito 2.
>
> **Reexecutado em 2026-07-26 com 202601 (D-29).** Nove transições:
> 0,110 · 0,091 · 0,097 · 0,097 · 0,092 · 0,083 · 0,079 · 0,087 · 0,097, mediana
> **0,092**, amplitude 0,079–0,110. Ainda plana, veredito mantido.
>
> A transição nova só ficou legível depois de dois bugs que a série de nove
> escondia. A primeira medição dela deu taxa **1,94** — tabela inteira contada
> como substituída — porque o CNES alargou `co_tipo_equipamento` de `CHAR(1)` para
> `CHAR(2)` e `'1'` virou `'1 '` (D-30). E o resumo vinha com 393 de 396 pares
> porque o diff usava a lista de colunas de um lado só, e as colunas instáveis de
> D-20 derrubavam a transição inteira (D-31). Taxa perto de 1,0 é sinal de chave
> ou de formato, não de dado.


## 4. A trilha geográfica é viável?

A trilha 3 depende inteiramente de `nu_latitude` e `nu_longitude`.

**Esta seção tem história.** D-17 concluiu que o teto de cobertura era 57% e que
43% dos estabelecimentos jamais seriam nós. O número estava errado: a medição
rodou com seis das nove competências convertidas, e as três que faltavam eram as
de melhor cobertura. D-22 corrige. A célula de abertura deste notebook agora
afirma o número de snapshots justamente para que isso não se repita.

In [ ]:
def cobertura_geografica() -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not p.exists():
            continue
        linhas.append(con.execute(f'''
            SELECT '{periodo}' AS periodo,
                   COUNT(*) AS estabelecimentos,
                   COUNT(nu_latitude) AS com_coordenada,
                   SUM(CASE WHEN nu_latitude BETWEEN -34 AND 6
                             AND nu_longitude BETWEEN -74 AND -34
                             AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                            THEN 1 ELSE 0 END) AS plausivel
            FROM read_parquet('{p}') WHERE {FILTRO}
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    df["plausivel_%"] = (100 * df["plausivel"] / df["estabelecimentos"]).round(2)
    return df

geo = cobertura_geografica()
print(geo.to_string(index=False))

In [ ]:
# Cobertura acumulada: um estabelecimento posicionado em QUALQUER snapshot pode
# ser usado, porque D-15 trata a posição como invariante no tempo. É o teto real
# da trilha 3. `union_by_name` é obrigatório aqui — três tabelas têm colunas que
# somem e voltam entre competências (D-20).
arquivos = [str(PRIMARY_FOLDER / p / "tbEstabelecimento.parquet")
            for p in PERIODOS
            if (PRIMARY_FOLDER / p / "tbEstabelecimento.parquet").exists()]
lista = ", ".join(f"'{a}'" for a in arquivos)

acumulada = con.execute(f'''
    WITH sel AS (
        SELECT co_unidade, nu_latitude, nu_longitude
        FROM read_parquet([{lista}], union_by_name=true) WHERE {FILTRO}
    )
    SELECT COUNT(DISTINCT co_unidade) AS estabelecimentos,
           COUNT(DISTINCT CASE WHEN nu_latitude BETWEEN -34 AND 6
                                AND nu_longitude BETWEEN -74 AND -34
                                AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                               THEN co_unidade END) AS posicionaveis
    FROM sel
''').df()
acumulada["cobertura_acumulada_%"] = (
    100 * acumulada["posicionaveis"] / acumulada["estabelecimentos"]).round(2)
print(acumulada.to_string(index=False))
print(f"\nMelhor snapshot isolado: {geo['plausivel_%'].max():.2f}%")
print("Se a acumulada mal supera o melhor snapshot, o teto é estrutural: quem")
print("não tem coordenada hoje nunca teve. Foi o que se observou (D-17, D-22).")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(geo["periodo"], geo["plausivel_%"])
ax.axhline(float(acumulada["cobertura_acumulada_%"].iloc[0]), color="crimson",
           linestyle="--", label="acumulada (teto real)")
ax.set_title(f"Cobertura de coordenada plausível — recorte {RECORTE!r}")
ax.set_ylabel("% dos estabelecimentos")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

> **Veredito 4 — trilha geográfica. Viável, com 85,7% no estado.**
>
> | Snapshot | Município | Estado |
> |---|---|---|
> | 201701 | 0,5% | 1,1% |
> | 201901 | 3,8% | 26,4% |
> | 202001 | 33,8% | 71,7% |
> | 202201 | 57,6% | 79,9% |
> | 202501 | **74,7%** | **85,7%** |
>
> União das nove competências: **75,0%** no município, **85,7%** no estado.
>
> Há um degrau claro em 2020, coerente com o CNES ter passado a exigir
> geolocalização. A união acrescenta quase nada sobre o melhor snapshot isolado
> — 85,7% contra 85,7% —, então o teto é estrutural: quem não tem coordenada em
> 2025 nunca teve.
>
> **Correção registrada.** D-17 afirmou teto de 57% medindo seis de nove
> snapshots. D-22 corrige. A conclusão qualitativa sobreviveu; o nível não.
>
> **O que continua valendo.** D-15 trata a posição como invariante no tempo,
> tomada da observação mais antiga — não para elevar o teto, mas para dar nós à
> janela de treino de 2018 e 2019, onde a cobertura própria é de 1% a 26%.
> CEP-5 foi medido como substituto e rejeitado: raio mediano de 4,90 km contra
> 8,70 km de um controle embaralhado, informativo mas só duas vezes melhor que o
> azar. E o filtro de plausibilidade foi apertado, porque 1,2% das coordenadas
> caíam a até 197 km do centro numa cidade de 35 km.
>
> **Obrigação que permanece.** Comparar a trilha 3 com as outras duas exige o
> **mesmo subconjunto de nós**, senão a comparação mede diferença de amostra em
> vez de diferença de estrutura. `Previsao.mascara_de_entidades()` existe para
> isso.
>
> **Reexecutado em 2026-07-26 com 202601 (D-29).** A cobertura sobe de novo:
> **87,27%** em 202601 contra 85,67% em 202501 e 83,44% em 202401. A acumulada
> sobre as dez competências é **87,18%** — abaixo do melhor snapshot isolado
> porque o denominador inclui estabelecimentos que só existiram em competências
> antigas, sem coordenada. O teto continua estrutural e a tendência continua de
> alta lenta: quem entra no cadastro hoje entra georreferenciado, quem entrou
> antes de 2020 em geral não voltou a ser.


## 5. O filtro empírico sobre a série inteira

D-06 aplicou o filtro usando apenas 201701 e 202501. Aqui ele é reaplicado sobre
a série inteira: uma coluna é degenerada se está 100% nula, ou constante, em
**todos** os snapshots.

Note o `union_by_name=true`. Sem ele a leitura falha — três das 44 tabelas têm
colunas que somem e voltam entre competências, com 201901 anômala (D-20).

In [ ]:
def triagem_empirica() -> pd.DataFrame:
    linhas = []
    for tabela, colunas in schema.CNES_EXTRACT_COLUMNS.items():
        arquivos = [PRIMARY_FOLDER / p / f"{tabela}.parquet" for p in PERIODOS]
        arquivos = [a for a in arquivos if a.exists()]
        if not arquivos:
            linhas.append({"tabela": tabela, "coluna": "*",
                           "motivo": "tabela ausente em todos os snapshots"})
            continue
        lista = ", ".join(f"'{a}'" for a in arquivos)
        fonte = f"read_parquet([{lista}], union_by_name=true)"
        presentes = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM {fonte}").fetchall()}
        total = con.execute(f"SELECT COUNT(*) FROM {fonte}").fetchone()[0]
        if total == 0:
            linhas.append({"tabela": tabela, "coluna": "*",
                           "motivo": "vazia em todos os snapshots"})
            continue

        usaveis = [c for c in colunas if c in presentes]
        for c in colunas:
            if c not in presentes:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "ausente do Parquet"})
        if not usaveis:
            continue
        # Uma query por tabela, não uma por coluna: 44 tabelas x ~9 colunas x 9
        # snapshots seriam milhares de varreduras dos mesmos arquivos.
        agregados = ", ".join(
            f'COUNT("{c}") AS nn{i}, COUNT(DISTINCT "{c}") AS nd{i}'
            for i, c in enumerate(usaveis))
        r = con.execute(f"SELECT {agregados} FROM {fonte}").df().iloc[0]
        for i, c in enumerate(usaveis):
            if r[f"nn{i}"] == 0:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "100% nula em todos os snapshots"})
            elif r[f"nd{i}"] <= 1:
                linhas.append({"tabela": tabela, "coluna": c,
                               "motivo": "constante em todos os snapshots"})
    return pd.DataFrame(linhas)

rejeitadas = triagem_empirica()
uteis = sum(len(v) for v in schema.CNES_USEFUL_COLUMNS.values())
print(f"{len(rejeitadas)} rejeições sobre {len(PERIODOS)} snapshots, "
      f"de {uteis} colunas hoje `util`:\n")
if not rejeitadas.empty:
    print(rejeitadas.sort_values(["motivo", "tabela"]).to_string(index=False))
else:
    print("Nenhuma — o filtro já foi aplicado e o doc está em dia.")

In [ ]:
# O lado simétrico da triagem: coluna que existe no CSV e o doc não declara.
# A célula acima só examina o que o doc já lista, então nunca acharia isto —
# foi assim que quatro colunas do CSV passaram nove competências sem avaliação.
# Lê apenas o header de cada CSV dentro do ZIP; não descompacta nada.
import re
import zipfile

from src.config.paths import RAW_FOLDER


def normaliza(nome: str) -> str:
    """Mesma transformação que o `normalize_names=True` do DuckDB aplica."""
    return re.sub(r"[^a-z0-9_]", "", nome.strip().lower())


def headers_do_zip(periodo: str) -> dict[str, list[str]]:
    caminho = RAW_FOLDER / f"BASE_DE_DADOS_CNES_{periodo}.ZIP"
    if not caminho.exists():
        return {}
    saida = {}
    with zipfile.ZipFile(caminho) as z:
        for info in z.infolist():
            nome = Path(info.filename).name
            if not nome.lower().endswith(".csv"):
                continue
            base = nome[:-4]
            if not re.fullmatch(r".*\d{6}", base):   # o sufixo é a competência
                continue
            with z.open(info) as f:
                cabecalho = f.readline().decode("ISO_8859_1").rstrip("\r\n")
            saida[base[:-6]] = [normaliza(c) for c in cabecalho.split(";")]
    return saida


PERIODOS_ZIP = [p for p in PERIODOS if (RAW_FOLDER / f"BASE_DE_DADOS_CNES_{p}.ZIP").exists()]
headers = {p: headers_do_zip(p) for p in PERIODOS_ZIP}

nao_declaradas, doc_sem_csv = [], []
for tabela in schema.FACT_TABLES:
    declaradas = {c.nome for c in schema.tabela(tabela).colunas}
    materializadas = set(schema.CNES_EXTRACT_COLUMNS[tabela])
    for periodo in PERIODOS_ZIP:
        csv = headers[periodo].get(tabela)
        if csv is None:
            doc_sem_csv.append({"tabela": tabela, "periodo": periodo,
                                "obs": "tabela sem CSV nesta competência"})
            continue
        for c in csv:
            if c not in declaradas:
                nao_declaradas.append({"tabela": tabela, "coluna": c, "periodo": periodo})
        for c in materializadas - set(csv):
            doc_sem_csv.append({"tabela": tabela, "periodo": periodo,
                                "obs": f"coluna materializada ausente: {c}"})

print(f"colunas do CSV não declaradas no doc: {len(nao_declaradas)}")
if nao_declaradas:
    print(pd.DataFrame(nao_declaradas)
          .groupby(["tabela", "coluna"])["periodo"]
          .agg(["count", "min", "max"]).to_string())

print(f"\ncolunas `util`/`pendente` do doc ausentes de algum CSV: {len(doc_sem_csv)}")
if doc_sem_csv:
    print(pd.DataFrame(doc_sem_csv).groupby(["tabela", "obs"])["periodo"]
          .agg(["count", "min", "max"]).to_string())
    print("\nAusência intermitente é o padrão de D-20, não erro do doc: "
          "`to_parquet` projeta só o que existe em cada competência.")


> **Veredito 5 — filtro empírico. Uma rejeição.**
>
> `rlEstabUnidAcolhim.tp_sus_nao_sus`, constante em toda a série, reclassificada
> para `descartada`. Colunas `util`: 389 para 388.
>
> O resultado importa mais pelo que **não** mudou: ampliar de duas para nove
> competências acrescentou uma única rejeição. O crivo de D-06 é estável, não
> aperta indefinidamente conforme se olha mais dado.
>
> Rodar esta célula de novo agora deve devolver zero rejeições, porque o doc já
> foi corrigido. Se voltar a apontar algo, é sinal de que o doc e os dados
> divergiram.
>
> **Achado colateral, virou D-20.** A leitura conjunta dos nove snapshots falhou
> na primeira tentativa: três das 44 tabelas têm colunas que somem e voltam, e o
> padrão aponta 201901 como competência anômala em vez de evolução progressiva do
> schema.
>
> **Reexecutado em 2026-07-26: zero rejeições**, como previsto. Mas esta célula
> só olha o que o doc já declara, e por isso não vê o buraco simétrico: colunas
> que **existem no CSV e não estão no doc**. Conferindo os headers dos nove ZIPs
> apareceram quatro — `tbEstabelecimento.co_tipo_abrangencia` e `st_coworking`
> (admitidas como `util`), `tbDadosProfissionalSus.no_social` (100% nula nas três
> competências em que existe, `descartada`) e `tbCargaHorariaSus.nu_cnpj_det_vinc`,
> que no CSV se chama `nu_cnpj_detalhamento_vinculo`. Contagem de `util`: 388 para
> 390. As três novas do lado do `tbEstabelecimento` também somem e voltam, então
> são D-20 e não schema crescendo.


## 6. A tarefa secundária de regressão é viável?

D-02 registrou a regressão da quantidade como **tarefa secundária**, sem nunca
medir se ela sustenta um experimento. Esta seção mede, e a pergunta precisa ser
formulada com cuidado, porque há três alvos diferentes escondidos na mesma frase:

| Formulação | Alvo | Piso trivial |
|---|---|---|
| **Nível** | `qt_existente` em `t+1` | persistência: repetir a quantidade de `t` |
| **Variação** | `qt_existente(t+1) − qt_existente(t)` | prever zero |
| **Variação no SUS** | o mesmo sobre a parcela disponível ao SUS | prever zero |

A terceira é a que o pedido menciona explicitamente, e tem um obstáculo de dado:
`qt_sus` **só existe em 202601** (D-29), então não há série. O que existe nas dez
competências é o sinalizador `tp_sus` por linha, que permite um proxy — somar
`qt_existente` das linhas marcadas como SUS.

**O precedente que orienta a leitura.** D-03 rejeitou a taxa de utilização
(`qt_uso / qt_existente`) por degeneração: as duas colunas têm moda 1, em 69,6% e
69,0% das linhas, e a razão é 1,0 na maioria dos casos. O risco aqui é o mesmo, e
a métrica que denuncia é a comparação com o piso trivial: se prever zero já dá
erro quase nulo, a variância a explicar não existe.

A medição abaixo tem que responder três coisas:

1. **Quantos pares persistentes mudam de quantidade** entre `t` e `t+1`. Se for
   1% ou 2%, o alvo é quase todo zero, e a tarefa é de detecção rara antes de ser
   de regressão.
2. **Qual a dispersão entre os que mudam.** Delta concentrado em ±1 significa
   pouco a modelar mesmo restringindo aos que se movem.
3. **Se o piso trivial é batível.** RMSE e MAE de prever zero, contra o desvio
   padrão do delta. Se as duas quantidades quase coincidem, não há sinal.

Uma quarta pergunta, de desenho: agregar por estabelecimento — o **total** de
equipamentos da unidade — dá um alvo com mais variância que o par
(estabelecimento, tipo), ao custo de perder a dimensão de item que a tarefa
primária usa. Vale medir os dois.

In [ ]:
# Onde a quantidade disponível ao SUS pode ser medida, e onde não pode.
TABELA_ALVO = "rlEstabEquipamento"

disponibilidade = []
for periodo in PERIODOS:
    p = PRIMARY_FOLDER / periodo / f"{TABELA_ALVO}.parquet"
    if not p.exists():
        continue
    colunas = {r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{p}')").fetchall()}
    disponibilidade.append({
        "periodo": periodo,
        "qt_existente": "qt_existente" in colunas,
        "qt_uso": "qt_uso" in colunas,
        "qt_sus": "qt_sus" in colunas,      # nova em 202601 (D-29)
        "tp_sus": "tp_sus" in colunas,      # sinalizador, presente em toda a série
    })
print(pd.DataFrame(disponibilidade).to_string(index=False))
print("\nqt_sus em uma única competência não sustenta série: sem dois instantes,")
print("não há variação a prever. O proxy é somar qt_existente por tp_sus.")

# Distribuição das quantidades na competência mais recente, para dimensionar a
# variância disponível *entre* pares — o teto do que qualquer modelo poderia
# explicar no alvo de nível.
f = PRIMARY_FOLDER / PERIODOS[-1] / f"{TABELA_ALVO}.parquet"
raiz = PRIMARY_FOLDER / PERIODOS[-1] / "tbEstabelecimento.parquet"
print(f"\nDistribuição em {PERIODOS[-1]}, recorte {RECORTE!r}:")
print(con.execute(f"""
    WITH sel AS (SELECT DISTINCT co_unidade FROM read_parquet('{raiz}') WHERE {FILTRO})
    SELECT COUNT(*) AS linhas,
           ROUND(AVG(qt_existente), 3) AS media_exist,
           ROUND(STDDEV(qt_existente), 3) AS sd_exist,
           MEDIAN(qt_existente) AS mediana_exist,
           MAX(qt_existente) AS max_exist,
           ROUND(100.0 * SUM(CASE WHEN qt_existente = 1 THEN 1 ELSE 0 END) / COUNT(*), 1)
               AS pct_exist_igual_1,
           ROUND(100.0 * SUM(CASE WHEN qt_uso = qt_existente THEN 1 ELSE 0 END) / COUNT(*), 1)
               AS pct_uso_igual_exist
    FROM read_parquet('{f}') JOIN sel USING (co_unidade)
""").df().T.to_string(header=False))


In [ ]:
# A medição decisiva: variação por par persistente, e o piso trivial de prever zero.
#
# Só pares presentes nos DOIS snapshots entram. Par que aparece é aquisição, que é
# a tarefa primária; par que desaparece é remoção. A tarefa secundária pergunta
# outra coisa: dado que o equipamento já está lá, a quantidade se move?
CHAVE_PAR = ("co_unidade", "co_equipamento", "co_tipo_equipamento")


def variacao_por_par(coluna: str = "qt_existente") -> pd.DataFrame:
    chave = ", ".join(CHAVE_PAR)
    linhas = []
    for t in changes.transicoes(PERIODOS):
        a = PRIMARY_FOLDER / t.origem / f"{TABELA_ALVO}.parquet"
        b = PRIMARY_FOLDER / t.destino / f"{TABELA_ALVO}.parquet"
        raiz = PRIMARY_FOLDER / t.origem / "tbEstabelecimento.parquet"
        if not (a.exists() and b.exists() and raiz.exists()):
            continue
        linhas.append(con.execute(f"""
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}') WHERE {FILTRO}
            ),
            j AS (
                SELECT x."{coluna}" AS q0, y."{coluna}" AS q1
                FROM read_parquet('{a}') x
                JOIN read_parquet('{b}') y USING ({chave})
                JOIN sel s ON s.co_unidade = x.co_unidade
            )
            SELECT '{t.destino}' AS transicao,
                   COUNT(*) AS pares,
                   SUM(CASE WHEN q1 <> q0 THEN 1 ELSE 0 END) AS mudaram,
                   AVG(q1 - q0) AS media_delta,
                   STDDEV(q1 - q0) AS sd_delta,
                   MIN(q1 - q0) AS min_delta,
                   MAX(q1 - q0) AS max_delta,
                   SQRT(AVG((q1 - q0) * (q1 - q0))) AS rmse_prever_zero,
                   AVG(ABS(q1 - q0)) AS mae_prever_zero,
                   STDDEV(q1 - q0) FILTER (WHERE q1 <> q0) AS sd_entre_os_que_mudaram
            FROM j
        """).df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if not df.empty:
        df["pct_mudaram"] = (100 * df.mudaram / df.pares).round(3)
    return df


for coluna in ("qt_existente", "qt_uso"):
    print(f"\n{'=' * 78}\nvariação de {coluna} por par persistente\n{'=' * 78}")
    print(variacao_por_par(coluna)[
        ["transicao", "pares", "mudaram", "pct_mudaram", "media_delta", "sd_delta",
         "min_delta", "max_delta", "rmse_prever_zero", "mae_prever_zero",
         "sd_entre_os_que_mudaram"]
    ].to_string(index=False))

print("\nLeitura: se `pct_mudaram` for de poucos por cento e `rmse_prever_zero`")
print("praticamente igual a `sd_delta`, prever zero é quase ótimo — e a tarefa é")
print("de detecção rara, não de regressão. É o mesmo diagnóstico de D-03.")


In [ ]:
# Onde a variação se concentra, e as duas alternativas de desenho:
#   (a) agregar por estabelecimento — mais variância, sem a dimensão de item;
#   (b) restringir ao SUS via tp_sus — o proxy para o alvo que qt_sus não sustenta.
ultima = changes.transicoes(PERIODOS)[-1]
a = PRIMARY_FOLDER / ultima.origem / f"{TABELA_ALVO}.parquet"
b = PRIMARY_FOLDER / ultima.destino / f"{TABELA_ALVO}.parquet"
raiz = PRIMARY_FOLDER / ultima.origem / "tbEstabelecimento.parquet"
chave = ", ".join(CHAVE_PAR)

print(f"Concentração do delta em {ultima} (recorte {RECORTE!r}):")
print(con.execute(f"""
    WITH sel AS (SELECT DISTINCT co_unidade FROM read_parquet('{raiz}') WHERE {FILTRO}),
    j AS (
        SELECT y.qt_existente - x.qt_existente AS delta
        FROM read_parquet('{a}') x
        JOIN read_parquet('{b}') y USING ({chave})
        JOIN sel s ON s.co_unidade = x.co_unidade
    )
    SELECT delta, COUNT(*) AS pares,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS pct
    FROM j GROUP BY delta ORDER BY pares DESC LIMIT 10
""").df().to_string(index=False))


def variacao_agregada(filtro_sus: str = "") -> pd.DataFrame:
    """Delta do total de equipamentos por estabelecimento, com ou sem recorte SUS."""
    linhas = []
    for t in changes.transicoes(PERIODOS):
        a = PRIMARY_FOLDER / t.origem / f"{TABELA_ALVO}.parquet"
        b = PRIMARY_FOLDER / t.destino / f"{TABELA_ALVO}.parquet"
        raiz = PRIMARY_FOLDER / t.origem / "tbEstabelecimento.parquet"
        if not (a.exists() and b.exists() and raiz.exists()):
            continue
        onde = f"WHERE {filtro_sus}" if filtro_sus else ""
        linhas.append(con.execute(f"""
            WITH sel AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}') WHERE {FILTRO}
            ),
            t0 AS (SELECT co_unidade, SUM(qt_existente) AS total
                   FROM read_parquet('{a}') {onde} GROUP BY 1),
            t1 AS (SELECT co_unidade, SUM(qt_existente) AS total
                   FROM read_parquet('{b}') {onde} GROUP BY 1),
            j AS (
                SELECT t0.total AS q0, t1.total AS q1
                FROM t0 JOIN t1 USING (co_unidade) JOIN sel USING (co_unidade)
            )
            SELECT '{t.destino}' AS transicao,
                   COUNT(*) AS estabelecimentos,
                   ROUND(100.0 * SUM(CASE WHEN q1 <> q0 THEN 1 ELSE 0 END) / COUNT(*), 2)
                       AS pct_mudaram,
                   AVG(q0) AS media_total, STDDEV(q0) AS sd_total,
                   AVG(q1 - q0) AS media_delta, STDDEV(q1 - q0) AS sd_delta,
                   SQRT(AVG((q1 - q0) * (q1 - q0))) AS rmse_prever_zero,
                   AVG(ABS(q1 - q0)) AS mae_prever_zero
            FROM j
        """).df().iloc[0].to_dict())
    return pd.DataFrame(linhas)


print(f"\n{'=' * 78}\n(a) total de equipamentos por estabelecimento\n{'=' * 78}")
print(variacao_agregada().to_string(index=False))

print(f"\n{'=' * 78}\n(b) idem, só linhas disponíveis ao SUS (tp_sus = '1')\n{'=' * 78}")
print(variacao_agregada("tp_sus = '1'").to_string(index=False))

print("\nA comparação que decide o desenho: sd_delta do agregado contra sd_delta do")
print("par. Se o agregado tiver variância substancialmente maior, a formulação por")
print("estabelecimento é a única com sinal — ao custo de perder a dimensão de item.")


> **Veredito 6 — tarefa secundária de regressão. Rejeitada, pelo critério escrito antes.**
>
> Medido no recorte estadual, nove transições, sobre os pares que existem nos dois
> snapshots — par que aparece é aquisição, par que desaparece é remoção, e nenhum
> dos dois é a pergunta desta seção.
>
> **2.276.691 pares persistentes. 25.475 mudam de quantidade: 1,119%.**
>
> | Transição | Pares | Mudaram | % | sd(Δ) | RMSE de prever zero | MAE de prever zero |
> |---|---|---|---|---|---|---|
> | 201801 | 199.362 | 2.312 | 1,16% | 4,437 | **4,438** | 0,109 |
> | 201901 | 212.520 | 2.065 | 0,97% | 3,384 | **3,384** | 0,078 |
> | 202101 | 239.482 | 2.907 | 1,21% | 4,914 | **4,915** | 0,114 |
> | 202401 | 279.941 | 2.417 | 0,86% | 2,776 | **2,776** | 0,060 |
> | 202601 | 308.092 | 4.131 | 1,34% | 3,896 | **3,896** | 0,110 |
>
> **O primeiro critério fecha a decisão.** Prever zero tem RMSE igual ao desvio
> padrão do alvo até a terceira decimal, em todas as nove transições. Isso é
> definicional quando a média é quase zero, e é exatamente o sintoma de que **não
> há variância a explicar**: o erro absoluto médio de prever zero é de 0,06 a 0,13
> equipamento. Um modelo teria de bater isso.
>
> **O segundo critério também falha.** Ele exigia delta concentrado em ±1 *com*
> mais de 10% dos pares se movendo. A concentração existe — em 202601, 98,659% dos
> pares têm Δ = 0, e ±1 responde por 0,595% — mas o movimento é de 1,3%, uma ordem
> de grandeza abaixo do necessário. E entre os que se movem a cauda é grossa:
> |Δ| mediano 2, p90 de 15, p99 de **105,7**, máximo de 961.
>
> **O terceiro critério é o que quase passa, e não passa.** Agregar por
> estabelecimento eleva o desvio do delta de 2,8–4,9 para 11,2–25,9 — quatro a
> cinco vezes. Mas só 2,4% a 4,3% dos estabelecimentos mudam o total, o RMSE de
> prever zero continua igual ao desvio, e o MAE fica **abaixo de um equipamento**
> (0,39 a 0,80). Ganhar variância mudando a unidade de análise não cria sinal:
> move a mesma esparsidade para outro nível.
>
> **Achado colateral, e é sobre qualidade de dado.** O delta vai de −1.308 a
> +1.375 num único par (estabelecimento, tipo de equipamento) em um ano. Nenhum
> estabelecimento adquire mil e trezentos aparelhos de um tipo em doze meses: isso
> é correção de cadastro, não aquisição física. A pouca variância que existe está
> concentrada em ruído de registro — o que reforça a rejeição em vez de atenuá-la,
> porque um modelo treinado nesse alvo aprenderia a prever erro de digitação.
>
> **Sobre a variante no SUS, que era o pedido específico.** `qt_sus` existe em uma
> única competência, 202601, e lá já é **87,1% zeros**, com 25,3% de nulos e apenas
> 12,0% das linhas coincidindo com `qt_existente`. Mesmo quando 202701 chegar e
> houver duas observações, a série terá **uma** transição — insuficiente para a
> partição de D-08 — e partirá de um alvo mais degenerado que o total. O proxy por
> `tp_sus` é mensurável hoje (83.005 linhas marcadas como SUS no estado, contra
> 242.010 não-SUS), mas herda a mesma esparsidade de variação.
>
> **Conclusão.** A regressão da quantidade sai do escopo, pelo mesmo motivo que
> levou D-03 a descartar a taxa de utilização: o alvo é quase constante. O
> fenômeno de interesse — a quantidade se move — é raro, e a formulação correta de
> um evento raro é a que o projeto já usa: **classificação binária de aquisição**.
> Registrado em D-37, com o critério de decisão fixado antes da medição.


## Decisão final

Seis vereditos fechados. Registrado em `docs/03-decisoes.md`, D-18 a D-22, e o
sexto em D-37.

| Decisão | Status | Valor fixado | Evidência |
|---|---|---|---|
| D-01 alvo | fechada | `rlEstabEquipamento` | 34.571 aquisições; `rlEstabServClass` empata em volume mas é classificação administrativa |
| D-04 / D-10 densidade | fechada | nove snapshots anuais | taxa entre 0,082 e 0,112, série plana |
| D-06 filtro empírico | fechada | uma rejeição a mais | crivo estável de 2 para 9 competências |
| Chave natural | fechada | as duas hipóteses confirmadas | zero duplicatas em todos os snapshots |
| Trilha 3 geográfica | fechada | 85,7% no estado | D-22 corrige o 57% de D-17 |
| D-21 recorte | fechada | estado de São Paulo | 2,9x eventos, 645 municípios |
| D-37 regressão secundária | **rejeitada** | tarefa sai do escopo | 1,119% dos pares mudam; RMSE de prever zero = sd(Δ) nas nove transições |

**Achados que não estavam previstos:**

1. **A prevalência é severa e irredutível** — 0,047% no estado. Duas restrições
   do espaço de candidatos foram testadas e rejeitadas: por par (tipo de unidade,
   equipamento) corta só 1,5%; por estabelecimento já equipado corta 50% dos
   candidatos mas leva 33% dos positivos e excluiria a primeira aquisição, o
   evento mais relevante para política pública. **MAP@k** passa a ser a métrica
   de destaque. D-19.
2. **Trocar o recorte inverteu a liderança entre alvos.** Uma conclusão medida no
   município não sobreviveu à mudança de escala — lição de método além do
   resultado.
3. **A modelagem do grafo relacional estava errada.** Um nó por linha de tabela
   de fato dá 76 milhões de nós no estado e, pior, não cria vizinho compartilhado
   entre estabelecimentos com o mesmo equipamento. As tabelas do CNES são listas
   de arestas. D-25.

**Consequências aplicadas ao código:**

- `docs/01-selecao-tabelas.md` — `rlEstabUnidAcolhim.tp_sus_nao_sus` descartada
- `docs/02-metodologia.md` — MAP@k como métrica de destaque; recorte estadual
- `src/ml/graph.py` — `RECORTE_PADRAO = '35'`, recorte como prefixo IBGE
- `src/ml/gnn.py` — grafo por categoria, e corte anterior a todos os rótulos (D-25)
- `src/ml/tasks.py` — amostragem de negativos só no treino (D-23)